# 01 — Árbol de decisión C4.5

Este notebook desarrolla un clasificador propio basado en el algoritmo C4.5. Un **clasificador** aprende de casos cuya respuesta correcta ya se conoce y luego asigna una clase a casos nuevos. El modelo organiza lo aprendido como un árbol: la **raíz** es la primera decisión, los **nodos internos** preguntan por un atributo, las **ramas** representan sus posibles respuestas y las **hojas** contienen la distribución de clases con la que se produce una predicción.

Se implementan los cálculos de impureza, la selección de divisiones, la construcción recursiva, los controles para detener el crecimiento y la predicción. Se admiten atributos categóricos, numéricos y combinaciones de ambos, incluso con valores faltantes. Pandas se usa para organizar tablas; el aprendizaje del árbol se programa aquí sin un estimador de aprendizaje automático.

### Fundamentos y notación

Sea $D$ el **conjunto de casos de entrenamiento que llega al nodo que se está analizando**. En la raíz, $D$ contiene todo el dataset de entrenamiento; en los nodos siguientes contiene los casos que llegan por las ramas anteriores, con los pesos que les corresponden.

La relación entre los símbolos y los datos es la siguiente:

| Símbolo | Qué representa |
|---|---|
| $i$ | Identifica un caso de $D$, es decir, una fila de datos. Se usa para recorrer los casos; no tiene que coincidir con el valor del índice de pandas. |
| $y_i$ | La clase real o respuesta correcta del caso $i$, guardada en la columna `objetivo`, separada de los atributos. |
| $c$ | Una de las clases posibles a la puede pertenecer un caso. |
| $w_i$ | El peso que aporta el caso $i$ al nodo. Al comenzar el entrenamiento, cada caso tiene peso 1. |
| $W$ | La suma de los pesos de **todos** los casos de ese nodo: $W=\sum_{i\in D}w_i$. Se denomina masa total. |
| $p_c$ | La proporción del peso total que corresponde a la clase $c$. |

Para obtener $p_c$, se suman únicamente los pesos de los casos cuya clase real es $c$ y se divide por la suma de los pesos de todos los casos del nodo:

$$p_c=\frac{\displaystyle\sum_{i\in D:\,y_i=c}w_i}{\displaystyle\sum_{i\in D}w_i}=\frac{\text{masa de la clase }c}{W}.$$

La condición $y_i=c$ debajo de la sumatoria significa **«incluir solo los casos cuya clase sea igual a $c$»**. El numerador considera una clase; el denominador considera todas. Al sumar las proporciones de todas las clases se obtiene 1.

**Ejemplo con pesos unitarios:** si al nodo llegan cinco filas, tres de clase `A` y dos de clase `B`, todas con peso 1, entonces $W=5$, $p_A=3/5=0{,}6$ y $p_B=2/5=0{,}4$. En este caso, sumar pesos equivale a contar filas.

**Ejemplo con pesos fraccionarios:** si al nodo llegan dos casos de clase `A` con pesos 1 y 0,5, y un caso de clase `B` con peso 1, entonces $W=1+0{,}5+1=2{,}5$. Por lo tanto, $p_A=(1+0{,}5)/2{,}5=0{,}6$ y $p_B=1/2{,}5=0{,}4$. Hay tres filas representadas, pero su masa total es 2,5.

Hablar de **masa** permite contar correctamente el aporte de un caso cuando, por el valor de un atributo faltante, su peso se reparte entre varias ramas. Ese reparto no convierte el caso original en varios casos completos: sus aportes entre las ramas conservan su peso original.

La **entropía** mide la mezcla de clases. Vale cero si todos los casos pertenecen a la misma clase; cuanto más equilibradas estén las clases, mayor es la incertidumbre. Se mide en bits porque se usa logaritmo en base 2:

$$H(D)=-\sum_c p_c\log_2 p_c,\qquad 0\log_2 0=0.$$

Si un atributo $A$ divide el nodo en ramas $D_j$, su entropía posterior y su **ganancia de información** son:

$$H_A(D)=\sum_j\frac{W_j}{W}H(D_j),\qquad G(D,A)=H(D)-H_A(D).$$

La ganancia expresa cuánta incertidumbre elimina la división. Puede favorecer atributos con muchos valores distintos; por eso se compara la **razón de ganancia** o *Gain Ratio*, que la divide por la información de la partición o *Split Info*:

$$I(D,A)=-\sum_j\frac{W_j}{W}\log_2\frac{W_j}{W},\qquad R(D,A)=\frac{G(D,A)}{I(D,A)}.$$

Se elige la división con mayor $R$. Una división necesita al menos dos ramas conocidas con masa positiva. Si $I=0$, no aporta una separación y se descarta. Las funciones de cálculo devuelven cero como protección cuando el denominador es cero.

### Decisiones de diseño

- **Categorías:** cada valor conocido genera una rama. El atributo se retira de ese camino porque volver a preguntar por la misma categoría no separaría sus casos.
- **Números:** se ordenan los valores conocidos distintos $v_1<\cdots<v_r$ y se evalúan $r-1$ cortes, con $t=v_i$. Cada corte crea $A\leq t$ y $A>t$. Se conserva el atributo para poder dividir intervalos en niveles posteriores. No se convierten enteros grandes a flotantes para buscar el corte.
- **Tipos:** números reales se consideran continuos; texto, booleanos y columnas `category` se consideran categóricos. Un código numérico que representa una categoría debe recibirse como texto o `category`. Una columna completamente vacía no propone divisiones.
- **Empates:** entre atributos se conserva el primero por orden de columnas; entre cortes, el menor. Entre clases se elige la primera observada en el entrenamiento completo. No interviene aleatoriedad.
- **Aprendizaje local:** cada decisión usa únicamente los casos y pesos que llegan a ese nodo. La selección es voraz: elige la mejor división disponible sin explorar todos los árboles posibles.
- **Crecimiento:** se aplican límites configurables antes de crear hijos. Este desarrollo es una variante didáctica basada en C4.5; no reproduce todas las heurísticas de un sistema completo.

### Valores faltantes y probabilidades

Un valor ausente (`None`, `NaN`, `pd.NA` o `NaT`) no se inventa ni se trata como una categoría. Para evaluar $A$, se separan los casos conocidos $D_K$ de los desconocidos $D_U$, de masas $W_K$ y $W_U$. Se reduce la ganancia según cuánto respaldo conocido tiene el atributo:

$$G^*(D,A)=\frac{W_K}{W}\left[H(D_K)-\sum_j\frac{W_j}{W_K}H(D_j)\right].$$
$$I^*(D,A)=-\sum_j\frac{W_j}{W}\log_2\frac{W_j}{W}-\frac{W_U}{W}\log_2\frac{W_U}{W},\qquad R^*=G^*/I^*.$$

Los desconocidos participan como un grupo adicional en $I^*$, sin crear una rama. Durante la construcción, cada caso desconocido de peso $w_i$ aporta $w_iq_j$ al hijo $j$, donde $q_j=W_j/W_K$. Así se conserva la masa total y la masa de cada clase. Los pesos positivos se conservan sin redondearlos.

Para predecir, una hoja devuelve sus frecuencias ponderadas $P(c)=W_c/W$. Un atributo conocido sigue una rama; si falta, se combinan las distribuciones completas de los hijos:

$$P(c\mid x,\text{nodo})=\sum_j q_jP(c\mid x,\text{hijo}_j),\qquad \widehat y=\arg\max_c P(c\mid x).$$

Una categoría nueva devuelve la distribución del nodo alcanzado. Esto difiere de un faltante, que consulta varias ramas. Las probabilidades son frecuencias empíricas, sin calibración. Repartir faltantes puede aumentar mucho el trabajo; los límites de crecimiento permiten controlarlo.

### Entrada, salida y parámetros

`entrenar(caracteristicas, objetivos)` recibe dos DataFrames alineados. El primero contiene solo atributos y el segundo una columna `objetivo` sin faltantes. `predecir` devuelve un DataFrame con la columna `prediccion`; `predecir_probabilidades` devuelve una columna por clase en el orden de `classes_`. Ambos conservan índice, duplicados y orden, sin modificar la entrada. Se rechazan tablas vacías, columnas repetidas, infinitos, complejos y cambios de columnas o de significado numérico.

| Parámetro en español | Nombre compatible | Significado y valor inicial |
|---|---|---|
| `filtro_ganancia` | Mismo nombre | Opción experimental desactivada (`False`): elegir cortes numéricos por ganancia y filtrar atributos por ganancia media antes de comparar su razón. |
| `umbral_ganancia` | `threshold` | Detiene si $G^*$ es estrictamente menor que el umbral, en bits; inicialmente 0. |
| `profundidad_maxima` | `max_depth` | Profundidad máxima, con raíz en 0; `None` permite crecer sin ese límite. |
| `minimo_muestras_division` | `min_samples_split` | Masa mínima necesaria para dividir, real y finita de al menos 2; inicialmente 2. Sin faltantes coincide con la cantidad de casos. |
| `minima_razon_ganancia` | `min_gain_ratio` | Detiene si $R^*$ es estrictamente menor que este valor sin unidades; inicialmente 0. |

También se detiene ante una clase pura, atributos agotados o ausencia de una división válida. Los valores iniciales permiten atravesar una división de ganancia cero si otra decisión posterior aporta información. `threshold` y `min_gain_ratio` controlan cantidades distintas y no son intercambiables. Se conservan los nombres `fit`, `predict` y `predict_proba` como accesos compatibles a las operaciones en español.

[MEJORA OPCIONAL — HU-07: IMPLEMENTAR POSPODA PARA REEMPLAZAR SUBÁRBOLES YA CONSTRUIDOS POR HOJAS. ESTE MECANISMO NO ES OBLIGATORIO POR SEPARADO: EL CRITERIO ADMITE PREPODA Y/O POSPODA, Y EL ÁRBOL YA DISPONE DE PREPODA CONFIGURABLE.]

Los atributos compatibles `threshold`, `max_depth`, `min_samples_split` y `min_gain_ratio` permiten consultar o modificar los mismos valores que sus nombres en español. Al volver a entrenar se validan otra vez. La masa mínima admite fracciones porque los casos con datos ausentes pueden aportar pesos fraccionarios.

### Filtro experimental de ganancia

`filtro_ganancia=False` conserva el procedimiento explicado: cada corte y cada atributo compiten por razón de ganancia. Si se activa, primero se elige el corte numérico con mayor ganancia ajustada; después se calcula la ganancia media de las mejores particiones por atributo. Solo compiten las que alcanzan esa media (con tolerancia de redondeo de 10⁻¹²), y entre ellas gana la mayor razón de ganancia. Un empate mantiene el orden ya definido. Esta opción modifica el árbol aprendido y se registra como parámetro efectivo; no se activa automáticamente para mejorar un resultado de prueba. Su comparación corresponde a experimentos separados con la misma partición. No es pospoda ni garantiza una mejora de generalización.


In [1]:
# Herramientas para representar nodos y realizar sumas y logaritmos.
from dataclasses import dataclass, field
from math import fsum, isfinite, log2
from numbers import Integral, Real

# Tablas y comprobaciones de los tipos de sus columnas.
import pandas as pd
from pandas.api.types import is_bool_dtype, is_complex_dtype, is_numeric_dtype, is_scalar


def es_faltante(valor):
    """Reconoce si un valor individual representa información ausente.

    Entrada: el valor de un atributo de un caso. Devuelve True para None,
    NaN, pd.NA y NaT; False para otros valores. Un texto vacío no se
    interpreta automáticamente como ausencia. No cambia el valor recibido.
    """
    # Se comprueba primero que sea un valor individual, no una lista.
    return is_scalar(valor) and bool(pd.isna(valor))


## 1. Cálculos de impureza y evaluación de una división

Las funciones siguientes separan el cálculo en pasos: preparar pesos, sumar el aporte de cada clase, medir su mezcla y evaluar cuánto mejora al dividir. Las etiquetas y los pesos se relacionan por posición; las posiciones son internas al nodo y no se confunden con el índice de pandas.

`evaluar_particion` devuelve la ganancia ajustada, la información de partición, su cociente, las posiciones conocidas por rama, las posiciones faltantes y las proporciones que permitirán repartirlas. Las claves `split_info` y `gain_ratio` conservan los nombres habituales de esas dos medidas.


In [2]:
def preparar_pesos(etiquetas, pesos=None):
    """Prepara etiquetas y pesos alineados por su posición en el nodo.

    Entradas: etiquetas contiene las clases; pesos contiene un aporte por caso.
    Si pesos es None, asigna 1.0 a cada caso. Devuelve dos listas del mismo
    largo. Un peso puede ser fraccionario o cero, pero no negativo.
    Lanza ValueError ante longitudes distintas, booleanos, pesos no reales,
    infinitos o NaN. La alineación es por posición, no por índice de pandas.
    """
    # Cada caso vale uno, salvo que llegue con un aporte fraccionario.
    etiquetas = list(etiquetas)
    pesos = [1.0] * len(etiquetas) if pesos is None else list(pesos)
    if len(etiquetas) != len(pesos):
        raise ValueError("Cada etiqueta debe tener su peso correspondiente.")
    if any(not isinstance(peso, Real) or isinstance(peso, bool)
           or not isfinite(peso) or peso < 0 for peso in pesos):
        raise ValueError("Los pesos deben ser reales finitos y no negativos.")
    return etiquetas, pesos


def conteos_ponderados(etiquetas, pesos):
    """Suma el aporte positivo de cada clase y devuelve un diccionario.

    Recibe etiquetas y pesos ya preparados y alineados. Cada clave es una
    clase y su valor es la masa total de esa clase. Los pesos cero no aportan
    ni crean una clase en el resultado. Las copias fraccionarias suman su
    aporte real: no se cuentan como casos completos adicionales.
    La función supone que los pesos ya se validaron; no los comprueba otra vez.
    """
    # Agrupar antes de sumar reduce los errores de redondeo.
    aportes_por_clase = {}
    for clase, peso in zip(etiquetas, pesos):
        if peso > 0:
            aportes_por_clase.setdefault(clase, []).append(peso)
    return {clase: fsum(aportes) for clase, aportes in aportes_por_clase.items()}


def entropia_masas(masas):
    """Calcula la entropía de masas no negativas y finitas ya validadas.

    Recibe los aportes de clases o grupos y devuelve la incertidumbre en bits.
    Normaliza cada masa por el total. Los términos cero no contribuyen y una
    lista vacía o de masa total cero devuelve 0.0. Esta función interna supone
    masas válidas; no rechaza por sí misma todas las entradas incorrectas.
    """
    # Sin masa no hay mezcla que medir.
    masas = list(masas)
    masa_total = fsum(masas)
    if masa_total == 0:
        return 0.0
    return -fsum((masa / masa_total) * log2(masa / masa_total)
                 for masa in masas if masa > 0)


def entropia(etiquetas, pesos=None):
    """Calcula H(D), en bits, usando la masa de cada clase.

    Recibe las etiquetas del nodo y, opcionalmente, sus pesos por posición.
    Devuelve un número: cero si no hay masa o si una sola clase aporta masa.
    Las proporciones se calculan con pesos, incluso después de varios repartos
    por faltantes. La validación de pesos se delega a preparar_pesos.
    """
    # La entropía depende de proporciones de clases, no del número de copias.
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    return entropia_masas(conteos_ponderados(etiquetas, pesos).values())


def entropia_ponderada(grupos, pesos_grupos=None):
    """Promedia la entropía de las ramas según sus masas.

    Entradas: grupos contiene una lista de etiquetas por rama; pesos_grupos
    contiene los pesos correspondientes, o None para usar pesos unitarios.
    Devuelve la suma de la entropía de cada rama multiplicada por su proporción
    de masa. Si la masa total es cero, devuelve 0.0. Lanza ValueError si faltan
    listas de pesos, no coinciden sus largos o sus valores son inválidos.
    Al evaluar un atributo con nulos, los grupos recibidos contienen conocidos.
    """
    # Preparar una lista de pesos por rama y comprobar su correspondencia.
    grupos = [list(grupo) for grupo in grupos]
    pesos_grupos = ([[1.0] * len(grupo) for grupo in grupos]
                    if pesos_grupos is None else list(pesos_grupos))
    if len(grupos) != len(pesos_grupos):
        raise ValueError("Cada grupo debe tener sus pesos correspondientes.")
    grupos_preparados = [preparar_pesos(grupo, pesos)
                         for grupo, pesos in zip(grupos, pesos_grupos)]
    masa_total = fsum(fsum(pesos) for _, pesos in grupos_preparados)
    if masa_total == 0:
        return 0.0

    # Una rama con mayor masa tiene más influencia en el promedio.
    return fsum(fsum(pesos) / masa_total * entropia(grupo, pesos)
                for grupo, pesos in grupos_preparados)


def ganancia_informacion(etiquetas, grupos, pesos=None, pesos_grupos=None):
    """Resta la entropía posterior a la entropía inicial, en bits.

    Recibe etiquetas del conjunto, grupos de etiquetas por rama y sus pesos
    opcionales. Devuelve la reducción de incertidumbre. El llamador debe
    asegurar que los grupos cubran el mismo conjunto con los mismos aportes;
    esta función no comprueba esa cobertura. El ajuste por datos desconocidos
    se realiza en evaluar_particion, no en esta resta.
    """
    # Ambas entropías usan los mismos casos y aportes.
    return entropia(etiquetas, pesos) - entropia_ponderada(grupos, pesos_grupos)


def informacion_particion(grupos, pesos_grupos=None):
    """Calcula Split Info: la entropía de las masas de las ramas.

    Recibe etiquetas agrupadas por rama y pesos opcionales por grupo.
    Devuelve un número en bits; con una sola rama positiva o masa cero vale 0.
    Sin pesos utiliza el tamaño de cada grupo. No incorpora por sí misma
    casos desconocidos: evaluar_particion agrega su masa al cálculo completo.
    La cantidad de grupos y las listas de pesos deben corresponderse.
    """
    # Con pesos unitarios basta contar las filas de cada grupo.
    grupos = [list(grupo) for grupo in grupos]
    if pesos_grupos is None:
        return entropia_masas([len(grupo) for grupo in grupos])
    pesos_grupos = list(pesos_grupos)
    if len(grupos) != len(pesos_grupos):
        raise ValueError("Cada grupo debe tener sus pesos correspondientes.")
    return entropia_masas([fsum(preparar_pesos(grupo, pesos)[1])
                          for grupo, pesos in zip(grupos, pesos_grupos)])


def razon_ganancia(etiquetas, grupos, pesos=None, pesos_grupos=None):
    """Calcula Gain Ratio para una partición completa sin desconocidos.

    Recibe etiquetas, grupos por rama y sus pesos opcionales. Devuelve ganancia
    de información dividida por Split Info, sin unidades. Si el denominador
    es cero devuelve 0.0 como protección; eso no vuelve elegible la división.
    El llamador debe asegurar la cobertura de los grupos. Para atributos con
    faltantes, el árbol utiliza la evaluación ajustada de evaluar_particion.
    """
    # Evitar una división por cero cuando no existe separación útil.
    informacion = informacion_particion(grupos, pesos_grupos)
    if informacion == 0:
        return 0.0
    return ganancia_informacion(etiquetas, grupos, pesos, pesos_grupos) / informacion


def evaluar_particion(etiquetas, indices_por_rama, pesos=None, faltantes=()):
    """Evalúa una división conocida y ajusta sus medidas por los faltantes.

    Entradas:
        etiquetas: clases de todos los casos representados en el nodo.
        indices_por_rama: diccionario rama -> posiciones de casos conocidos.
            Las posiciones empiezan en cero dentro del nodo, no son sus índices
            de pandas. Cada conocido con peso positivo pertenece a una sola rama.
        pesos: aporte por posición; None equivale a uno por caso.
        faltantes: posiciones cuyo valor del atributo evaluado se desconoce.
            Deben quedar fuera de las ramas conocidas.

    El llamador debe aportar posiciones válidas, sin repetir ni superponer
    conocidos y desconocidos, y cubrir los casos con masa positiva. La función
    valida los pesos, pero no comprueba esas condiciones sobre las posiciones.

    Devuelve un diccionario con:
        entropia_ponderada: incertidumbre posterior solo de los conocidos.
        ganancia: mejora entre conocidos multiplicada por masa_conocida / total.
        split_info: entropía de las masas conocidas por rama y de la masa faltante.
        gain_ratio: ganancia ajustada / split_info; cero si el denominador es cero.
        ramas: posiciones conocidas por rama, para construir los descendientes.
        faltantes: lista de posiciones que luego deberán repartirse.
        proporciones: masa conocida de cada rama / masa conocida total.

    Si no hay masa conocida, la ganancia es cero y proporciones queda vacío.
    La masa desconocida participa en Split Info como grupo de cálculo, sin
    crear una rama. Esta función evalúa; no copia casos ni reparte sus pesos.
    Los buscadores descartan los atributos sin dos ramas conocidas positivas.
    """
    # Reunir las etiquetas y masas conocidas de cada rama.
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    grupos = [[etiquetas[posicion] for posicion in posiciones]
              for posiciones in indices_por_rama.values()]
    pesos_grupos = [[pesos[posicion] for posicion in posiciones]
                    for posiciones in indices_por_rama.values()]
    posiciones_conocidas = [posicion for posiciones in indices_por_rama.values()
                            for posicion in posiciones]
    masa_conocida = fsum(pesos[posicion] for posicion in posiciones_conocidas)
    masa_faltante = fsum(pesos[posicion] for posicion in faltantes)
    masa_total = masa_conocida + masa_faltante

    # Medir la mejora sobre conocidos y reducirla según su presencia en el nodo.
    ganancia_ajustada = 0.0
    if masa_total > 0 and masa_conocida > 0:
        ganancia_ajustada = masa_conocida / masa_total * ganancia_informacion(
            [etiquetas[posicion] for posicion in posiciones_conocidas], grupos,
            [pesos[posicion] for posicion in posiciones_conocidas], pesos_grupos)
    informacion = entropia_masas([fsum(pesos_rama) for pesos_rama in pesos_grupos]
                                 + [masa_faltante])

    # Guardar también lo necesario para construir los hijos sin recalcular cortes.
    return {
        "entropia_ponderada": entropia_ponderada(grupos, pesos_grupos),
        "ganancia": ganancia_ajustada,
        "split_info": informacion,
        "gain_ratio": ganancia_ajustada / informacion if informacion > 0 else 0.0,
        "ramas": indices_por_rama,
        "faltantes": list(faltantes),
        "proporciones": {
            rama: fsum(pesos[posicion] for posicion in posiciones) / masa_conocida
            for rama, posiciones in indices_por_rama.items()
        } if masa_conocida else {},
    }


# Nombres habituales conservados como accesos a las mismas funciones.
split_info = informacion_particion
gain_ratio = razon_ganancia


## 2. Búsqueda de divisiones categóricas y numéricas

La búsqueda separa primero los valores conocidos y faltantes. Las categorías agrupan las posiciones por valor; los números prueban todos los cortes entre valores distintos. Se excluyen casos de peso cero y atributos constantes. Recorrer los cortes en orden creciente y reemplazar el mejor solo ante una mejora estricta resuelve los empates de forma estable.

Esta búsqueda numérica privilegia la claridad: vuelve a evaluar las ramas de cada corte. Con $n$ casos y $r$ valores distintos cuesta aproximadamente $O(n\log n+rn)$ por atributo y nodo. No se utiliza una optimización por conteos acumulados.

### Qué significa cortar un atributo continuo

Un atributo continuo admite medidas con decimales. Para evaluar sus cortes se usan los valores conocidos distintos del nodo, sin convertirlos en categorías ni redondearlos. Se prueban todos los límites candidatos y se conserva **un corte por nodo** si ese atributo resulta elegido frente a los demás. El valor exactamente igual al corte sigue la rama `<=`. El mismo atributo puede aparecer en decisiones posteriores: cada nodo vuelve a buscar su corte solo con los casos que le llegan. Así se pueden distinguir varios intervalos mediante decisiones sucesivas.

En una tabla mixta, cada atributo conserva su tratamiento: ramas por categoría o dos ramas por corte numérico. Sus mejores candidatos se comparan con la misma razón de ganancia ajustada. Los números que representan códigos de categorías deben declararse como texto o `category` para evitar interpretarlos como medidas.


In [3]:
def particion_categorica(valores, etiquetas, pesos=None):
    """Agrupa los valores conocidos y evalúa una división por categoría.

    Recibe valores de un atributo, etiquetas y pesos opcionales alineados por
    posición. Los valores conocidos deben poder usarse como claves de un
    diccionario. Ignora pesos cero y separa las posiciones faltantes.
    Devuelve el diccionario de evaluar_particion, o None si quedan menos de
    dos categorías conocidas con masa positiva, incluso si hay muchos nulos.
    Lanza ValueError si no coincide el número de valores y etiquetas o si
    los pesos son inválidos. El entrenamiento valida antes las categorías.
    """
    # Comprobar que cada valor tenga una etiqueta y un peso.
    valores = list(valores)
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    if len(valores) != len(etiquetas):
        raise ValueError("Cada valor debe tener su etiqueta correspondiente.")

    # Guardar los ausentes aparte: luego su peso se repartirá entre los hijos.
    ramas, posiciones_faltantes = {}, []
    for posicion, (valor, peso) in enumerate(zip(valores, pesos)):
        if peso == 0:
            continue
        if es_faltante(valor):
            posiciones_faltantes.append(posicion)
        else:
            ramas.setdefault(valor, []).append(posicion)
    if len(ramas) < 2:
        return None
    return evaluar_particion(etiquetas, ramas, pesos, posiciones_faltantes)


def buscar_mejor_umbral(valores, etiquetas, pesos=None, por_ganancia=False):
    """Busca el corte binario de mayor razón de ganancia para un atributo numérico.

    Entradas: valores contiene las medidas del nodo; etiquetas, sus clases;
    pesos, el aporte de cada caso, o None para usar uno por caso. Las tres
    secuencias se relacionan por posición. Los conocidos con peso positivo
    deben ser números reales finitos y no booleanos; se admiten faltantes.

    Ordena los conocidos distintos y prueba cada valor salvo el mayor como
    umbral t. No redondea los decimales ni calcula puntos medios. Cada candidato
    forma dos ramas no vacías: <= y >. Los valores repetidos permanecen juntos.
    Los faltantes no proponen cortes, pero sí ajustan las medidas de evaluación.

    Devuelve los campos de evaluar_particion más umbral, el valor observado
    elegido. Devuelve None si no hay dos valores conocidos distintos con peso
    positivo: atributo vacío, constante o sin masa suficiente para separar.
    Los pesos cero se ignoran antes de inspeccionar sus valores.

    Lanza ValueError ante longitudes incompatibles, pesos inválidos o conocidos
    numéricos inválidos con peso positivo. Ante Gain Ratio exactamente igual,
    conserva el menor corte porque recorre candidatos en orden creciente.
    Con por_ganancia=True selecciona por ganancia ajustada en lugar del cociente.
    Busca un corte para este nodo; no decide la prepoda ni elimina el atributo,
    que puede volver a dividirse en nodos posteriores.
    """
    # Validar los conocidos y conservar sus posiciones dentro del nodo.
    valores = list(valores)
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    if len(valores) != len(etiquetas):
        raise ValueError("Cada valor debe tener su etiqueta correspondiente.")
    posiciones_conocidas, posiciones_faltantes = [], []
    for posicion, (valor, peso) in enumerate(zip(valores, pesos)):
        if peso == 0:
            continue
        if es_faltante(valor):
            posiciones_faltantes.append(posicion)
        elif not isinstance(valor, Real) or isinstance(valor, bool) or not isfinite(valor):
            raise ValueError("Los valores numéricos conocidos deben ser reales y finitos.")
        else:
            posiciones_conocidas.append(posicion)

    # El mayor valor se excluye porque dejaría vacía la rama derecha.
    umbrales_candidatos = sorted({valores[posicion]
                                 for posicion in posiciones_conocidas})[:-1]
    mejor_particion = None
    for umbral in umbrales_candidatos:
        ramas = {
            "<=": [posicion for posicion in posiciones_conocidas if valores[posicion] <= umbral],
            ">": [posicion for posicion in posiciones_conocidas if valores[posicion] > umbral],
        }
        evaluacion = evaluar_particion(etiquetas, ramas, pesos, posiciones_faltantes)
        evaluacion["umbral"] = umbral
        # El criterio alternativo se usa únicamente con el filtro experimental.
        medida_seleccion = "ganancia" if por_ganancia else "gain_ratio"
        if (mejor_particion is None
                or evaluacion[medida_seleccion] > mejor_particion[medida_seleccion]):
            mejor_particion = evaluacion
    return mejor_particion


## 3. Información que conserva cada nodo

Cada nodo guarda la clase más probable, los conteos por clase, la entropía y la masa total. `n_ejemplos` cuenta las filas representadas, mientras que `peso_total` suma sus aportes; con faltantes pueden ser cantidades diferentes. Una decisión guarda además el atributo, su tipo, el corte si corresponde, sus medidas y las proporciones de las ramas. Una hoja no tiene hijos y registra por qué se detuvo.


In [4]:
@dataclass
class Nodo:
    """Representa una decisión o una hoja con la información aprendida del nodo."""
    # Resumen de las clases y del aporte de los casos que llegaron hasta aquí.
    clase: object
    conteos: dict
    n_ejemplos: int
    entropia: float
    peso_total: float = 0.0

    # Decisión que conecta este nodo con sus hijos; queda vacía en las hojas.
    atributo: object = None
    tipo: str | None = None
    umbral: float | int | None = None
    hijos: dict = field(default_factory=dict)
    proporciones: dict = field(default_factory=dict)
    ganancia: float | None = None
    split_info: float | None = None
    gain_ratio: float | None = None
    motivo_parada: str | None = None

    @property
    def es_hoja(self):
        """Indica que no existen decisiones descendientes."""
        return not self.hijos


## 4. Entrenamiento, prepoda y predicción

El entrenamiento valida tablas y parámetros, recuerda el orden de clases y columnas e inicia todos los pesos en uno. La construcción crea el resumen del nodo, aplica las condiciones de parada y busca la mejor partición. Si puede dividir, construye cada hijo con sus casos conocidos y los aportes correspondientes de los faltantes. La profundidad aumenta en uno por nivel.

La inferencia sigue las decisiones aprendidas. Para conservar enteros grandes y tipos individuales se leen las filas como tuplas, evitando conversiones de tipo de una fila completa. La normalización final solo absorbe errores de redondeo de las sumas; no cambia el criterio de clasificación.

### Reparto de pesos e inferencia: dos momentos diferentes

Durante el **entrenamiento**, si falta el valor del atributo elegido en un caso, ese caso participa en todos los hijos con una fracción de su peso. Mantiene su clase real y sus demás atributos. Como las proporciones de las ramas suman uno, repartir el aporte conserva el peso original; si vuelve a repartirse más abajo, se multiplica el peso que ya recibió, sin reiniciarlo en uno. Los conocidos solo participan en su propia rama. La masa mínima para dividir se comprueba sobre estos aportes, no sobre la cantidad de copias representadas.

Durante la **predicción**, se desconoce la clase de la consulta. Si falta el atributo que se pregunta, se recorren las ramas y se combinan sus probabilidades de clase. La rama más frecuente no se toma como respuesta automática y el dato ausente no recibe un valor inventado. Solo al final se elige la clase con mayor probabilidad. Si el faltante pertenece a una columna que el nodo no consulta, no impide seguir su decisión normalmente.

### Casos borde y respuesta definida

| Situación | Respuesta del algoritmo |
|---|---|
| Valores numéricos repetidos | No generan cortes duplicados; valores iguales permanecen en la misma rama. |
| Un único valor conocido distinto o todos los valores ausentes | Ese atributo no propone una división. Si ninguno puede dividir, se crea una hoja con la distribución de clases del nodo. |
| Casos con peso cero | No proponen cortes ni aportan masa a las clases o ramas. |
| Consulta exactamente igual al umbral | Sigue la rama `<=`; un valor mayor sigue `>`. |
| Consulta numérica fuera del rango observado | Se compara normalmente con los umbrales aprendidos, sin inventar nuevos cortes. |
| Dos cortes o atributos con igual razón de ganancia | Se conserva el menor corte de un atributo y el primer atributo por orden de columnas. |
| Falta el atributo consultado | En entrenamiento se reparte peso; en predicción se mezclan las distribuciones completas de los hijos. |
| Categoría nueva en el nodo consultado | Se devuelve la distribución local del nodo; no se trata como un faltante. |
| Todos los atributos de una consulta están ausentes | Se combinan las rutas y se recuperan las proporciones de clases de la raíz, salvo redondeo. |
| Clases empatadas en la predicción | Se conserva la primera clase observada en el entrenamiento completo. |
| Texto vacío o un marcador textual de ausencia | Se interpreta como categoría; debe convertirse a una representación de faltante si realmente significa ausencia. |
| Objetivo faltante, infinito, número complejo o estructura de entrada inválida | Se rechaza la entrada; los nulos admitidos corresponden a atributos, no a respuestas correctas. |

Las funciones internas de evaluación reciben posiciones preparadas por los buscadores. Suponen que los grupos conocidos son disjuntos, que las posiciones faltantes no se repiten en ellos y que todos los casos de masa positiva están representados. Estas condiciones se describen como requisitos de uso, sin afirmar que la función interna vuelva a comprobarlas todas.


In [5]:
class ArbolC45:
    """Árbol propio con categorías, cortes numéricos, nulos y prepoda.

    Los cuatro parámetros en español controlan el crecimiento. Se admiten
    sus nombres compatibles en inglés, sin permitir configuraciones ambiguas.
    """

    def __init__(self, umbral_ganancia=0.0, profundidad_maxima=None,
                 minimo_muestras_division=2, minima_razon_ganancia=0.0,
                 filtro_ganancia=False, **parametros_compatibles):
        # Traducir los nombres compatibles sin mantener dos implementaciones.
        parametros = dict(umbral_ganancia=umbral_ganancia,
                          profundidad_maxima=profundidad_maxima,
                          minimo_muestras_division=minimo_muestras_division,
                          minima_razon_ganancia=minima_razon_ganancia)
        equivalencias = {
            "threshold": ("umbral_ganancia", 0.0),
            "max_depth": ("profundidad_maxima", None),
            "min_samples_split": ("minimo_muestras_division", 2),
            "min_gain_ratio": ("minima_razon_ganancia", 0.0),
        }
        for nombre, valor in parametros_compatibles.items():
            if nombre not in equivalencias:
                raise TypeError(f"Parámetro desconocido: {nombre}.")
            nombre_espanol, valor_inicial = equivalencias[nombre]
            if parametros[nombre_espanol] != valor_inicial:
                raise ValueError(f"Usá solo un nombre para {nombre_espanol}.")
            parametros[nombre_espanol] = valor
        for nombre, valor in parametros.items():
            setattr(self, nombre, valor)
        self.filtro_ganancia = filtro_ganancia
        self.raiz_ = None
        self._validar_parametros()

    # Los nombres compatibles reflejan los mismos parámetros en español.
    @property
    def threshold(self):
        """Acceso compatible al valor de umbral_ganancia."""
        return self.umbral_ganancia

    @threshold.setter
    def threshold(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.umbral_ganancia = valor

    @property
    def max_depth(self):
        """Acceso compatible al valor de profundidad_maxima."""
        return self.profundidad_maxima

    @max_depth.setter
    def max_depth(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.profundidad_maxima = valor

    @property
    def min_samples_split(self):
        """Acceso compatible al valor de minimo_muestras_division."""
        return self.minimo_muestras_division

    @min_samples_split.setter
    def min_samples_split(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.minimo_muestras_division = valor

    @property
    def min_gain_ratio(self):
        """Acceso compatible al valor de minima_razon_ganancia."""
        return self.minima_razon_ganancia

    @min_gain_ratio.setter
    def min_gain_ratio(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.minima_razon_ganancia = valor

    def obtener_parametros(self):
        """Devuelve los valores efectivos para registrar cómo se construyó el árbol."""
        return {nombre: getattr(self, nombre) for nombre in (
            "umbral_ganancia", "profundidad_maxima",
            "minimo_muestras_division", "minima_razon_ganancia", "filtro_ganancia")}

    def _validar_parametros(self):
        """Rechaza límites negativos, tipos incorrectos y valores no finitos."""
        if not isinstance(self.filtro_ganancia, bool):
            raise ValueError("filtro_ganancia debe ser True o False.")
        # Los dos umbrales son números reales; la profundidad es un entero y la masa puede ser fraccionaria.
        for nombre in ("umbral_ganancia", "minima_razon_ganancia"):
            valor = getattr(self, nombre)
            if (not isinstance(valor, Real) or isinstance(valor, bool)
                    or not isfinite(valor) or valor < 0):
                raise ValueError(f"{nombre} debe ser real, finito y no negativo.")
        if self.profundidad_maxima is not None and (
                not isinstance(self.profundidad_maxima, Integral)
                or isinstance(self.profundidad_maxima, bool) or self.profundidad_maxima < 0):
            raise ValueError("profundidad_maxima debe ser None o un entero no negativo.")
        if (not isinstance(self.minimo_muestras_division, Real)
                or isinstance(self.minimo_muestras_division, bool)
                or not isfinite(self.minimo_muestras_division)
                or self.minimo_muestras_division < 2):
            raise ValueError("minimo_muestras_division debe ser real, finito y de al menos 2.")

    @staticmethod
    def _validar_caracteristicas(caracteristicas):
        """Comprueba la tabla de entrada sin modificarla; admite datos ausentes."""
        # Comprobar primero el formato general y los nombres de los atributos.
        if not isinstance(caracteristicas, pd.DataFrame):
            raise TypeError("Las características deben ser un pandas.DataFrame.")
        if caracteristicas.empty:
            raise ValueError("La tabla debe contener casos y atributos.")
        if not caracteristicas.columns.is_unique or caracteristicas.columns.isna().any():
            raise ValueError("Los nombres de los atributos deben ser únicos y no nulos.")

        # Los valores conocidos deben poder compararse o usarse como categorías.
        for columna in caracteristicas.columns:
            serie = caracteristicas[columna]
            if is_complex_dtype(serie.dtype):
                raise ValueError("No se admiten atributos de números complejos.")
            for valor in serie:
                if es_faltante(valor):
                    continue
                if isinstance(valor, complex) or (isinstance(valor, Real) and not isfinite(valor)):
                    raise ValueError("Los números conocidos deben ser reales y finitos.")
                try:
                    hash(valor)
                except TypeError as error:
                    raise ValueError("Las categorías deben ser valores individuales comparables.") from error

    def entrenar(self, caracteristicas, objetivos):
        """Aprende el árbol con dos DataFrames alineados y devuelve este modelo.

        objetivos debe tener una columna objetivo sin nulos. El entrenamiento
        reemplaza cualquier árbol anterior y nunca modifica las tablas recibidas.
        """
        # Validar las entradas antes de establecer el estado del modelo.
        self._validar_parametros()
        self._validar_caracteristicas(caracteristicas)
        if not isinstance(objetivos, pd.DataFrame):
            raise TypeError("Los objetivos deben ser un pandas.DataFrame.")
        if (list(objetivos.columns) != ["objetivo"]
                or not caracteristicas.index.equals(objetivos.index)):
            raise ValueError("objetivos debe tener la columna objetivo y el mismo índice y orden.")
        if objetivos.isna().any().any():
            raise ValueError("Las respuestas correctas no deben contener faltantes.")
        etiquetas = objetivos["objetivo"].tolist()
        try:
            clases = list(dict.fromkeys(etiquetas))
        except TypeError as error:
            raise ValueError("Las clases deben ser valores individuales comparables.") from error

        # Recordar el significado de las columnas y un orden estable de clases.
        self.classes_ = pd.Index(clases, tupleize_cols=False)
        self.columnas_ = caracteristicas.columns.copy()
        self.tipos_ = {
            columna: ("numerico" if is_numeric_dtype(caracteristicas[columna].dtype)
                      and not is_bool_dtype(caracteristicas[columna].dtype) else "categorico")
            for columna in caracteristicas.columns
        }
        self.raiz_ = self._construir(caracteristicas, etiquetas,
                                     list(caracteristicas.columns), [1.0] * len(etiquetas), 0)
        return self

    def _construir(self, caracteristicas, etiquetas, atributos, pesos, profundidad):
        """Construye y devuelve el Nodo raíz de un subárbol, con sus hijos si corresponde.

        Entradas internas ya preparadas por el entrenamiento:
            caracteristicas: filas que llegan a este nodo, con sus columnas originales.
            etiquetas y pesos: listas alineadas con esas filas por posición.
            atributos: columnas que todavía pueden utilizarse en este camino.
            profundidad: nivel actual, con la raíz en cero.
        Se presupone masa positiva y se conserva el orden global de clases.

        Primero resume las masas y aplica las paradas por pureza, atributos, nivel
        o masa mínima. Compara particiones categóricas y numéricas mediante Gain
        Ratio ajustado y aplica los umbrales de ganancia y razón por separado.
        Si se detiene devuelve una hoja con motivo_parada y distribución de clases.

        Al dividir, cada conocido entra únicamente en su rama con el peso recibido.
        Cada desconocido participa en todos los hijos con peso original * proporción
        de la rama, sin cambiar su clase ni sus otros atributos. Se conservan los
        aportes positivos y el orden; los índices de pandas no deciden el reparto.
        La suma de pesos de los hijos conserva la masa del padre y la de cada clase,
        salvo el redondeo numérico de las operaciones.

        Retira del camino el atributo categórico elegido y conserva el numérico,
        permitiendo nuevos cortes. Cada llamada descendiente aumenta un nivel.
        No modifica las tablas originales ni aprende de casos de validación o prueba.
        """
        # Resumir las clases con el orden global para resolver sus empates.
        masas_clases = conteos_ponderados(etiquetas, pesos)
        conteos = {clase: masas_clases[clase] for clase in self.classes_ if clase in masas_clases}
        clase_mayoritaria = max(conteos, key=conteos.get)
        nodo = Nodo(clase_mayoritaria, conteos, len(etiquetas), entropia(etiquetas, pesos),
                    peso_total=fsum(conteos.values()))

        # Detenerse cuando ya no hace falta o no está permitido seguir dividiendo.
        if len(conteos) == 1:
            nodo.motivo_parada = "clase_pura"
        elif not atributos:
            nodo.motivo_parada = "sin_atributos"
        elif self.profundidad_maxima is not None and profundidad >= self.profundidad_maxima:
            nodo.motivo_parada = "profundidad_maxima"
        elif nodo.peso_total < self.minimo_muestras_division:
            nodo.motivo_parada = "masa_menor_que_min_samples_split"
        if nodo.motivo_parada is not None:
            return nodo

        # Comparar las mejores divisiones de todos los atributos disponibles.
        candidatas = []
        for atributo in atributos:
            valores = caracteristicas[atributo].tolist()
            candidata = (buscar_mejor_umbral(valores, etiquetas, pesos, self.filtro_ganancia)
                         if self.tipos_[atributo] == "numerico"
                         else particion_categorica(valores, etiquetas, pesos))
            if candidata is None or candidata["split_info"] == 0:
                continue
            candidatas.append(dict(candidata, atributo=atributo))

        # El filtro opcional exige una ganancia al menos igual al promedio.
        # Se compara con una tolerancia pequeña para absorber redondeo numérico.
        if self.filtro_ganancia and candidatas:
            ganancia_media = fsum(candidata["ganancia"] for candidata in candidatas) / len(candidatas)
            candidatas = [candidata for candidata in candidatas
                          if candidata["ganancia"] >= ganancia_media - 1e-12]
        mejor_particion = None
        for candidata in candidatas:
            if mejor_particion is None or candidata["gain_ratio"] > mejor_particion["gain_ratio"]:
                mejor_particion = candidata
        if mejor_particion is None:
            nodo.motivo_parada = "sin_particion_valida"
            return nodo

        # Ganancia en bits y razón de ganancia tienen controles independientes.
        if mejor_particion["ganancia"] < self.umbral_ganancia:
            nodo.motivo_parada = "ganancia_menor_que_threshold"
            return nodo
        if mejor_particion["gain_ratio"] < self.minima_razon_ganancia:
            nodo.motivo_parada = "gain_ratio_menor_que_min_gain_ratio"
            return nodo

        # Guardar la decisión elegida antes de construir sus ramas.
        atributo = mejor_particion["atributo"]
        nodo.atributo = atributo
        nodo.tipo = self.tipos_[atributo]
        nodo.umbral = mejor_particion.get("umbral")
        nodo.ganancia = mejor_particion["ganancia"]
        nodo.split_info = mejor_particion["split_info"]
        nodo.gain_ratio = mejor_particion["gain_ratio"]
        nodo.proporciones = mejor_particion["proporciones"]
        atributos_siguientes = ([nombre for nombre in atributos if nombre != atributo]
                                if nodo.tipo == "categorico" else atributos)

        # Cada hijo recibe sus conocidos y la fracción correspondiente de los ausentes.
        posiciones_faltantes = set(mejor_particion["faltantes"])
        for rama, posiciones_conocidas in mejor_particion["ramas"].items():
            posiciones_conocidas = set(posiciones_conocidas)
            posiciones_hijo, pesos_hijo = [], []
            for posicion, peso in enumerate(pesos):
                nuevo_peso = peso if posicion in posiciones_conocidas else (
                    peso * nodo.proporciones[rama] if posicion in posiciones_faltantes else 0.0)
                if nuevo_peso > 0:
                    posiciones_hijo.append(posicion)
                    pesos_hijo.append(nuevo_peso)
            nodo.hijos[rama] = self._construir(
                caracteristicas.iloc[posiciones_hijo],
                [etiquetas[posicion] for posicion in posiciones_hijo],
                atributos_siguientes, pesos_hijo, profundidad + 1)
        return nodo

    def _distribucion_fila(self, fila, nodo):
        """Devuelve una lista de probabilidades de clase para una fila desde un nodo.

        Recibe fila, un diccionario atributo -> valor ya validado, y nodo, un Nodo
        aprendido con masa positiva. La lista devuelta sigue el orden global de
        classes_, incluyendo cero para clases sin masa en la hoja o nodo consultado.

        Una hoja devuelve sus frecuencias ponderadas. Un valor conocido sigue una
        rama: <= incluye el umbral y > los valores superiores. Si falta el atributo
        consultado, recorre todos los hijos y suma sus distribuciones multiplicadas
        por las proporciones aprendidas. No elige la rama más frecuente ni vota solo
        las clases ganadoras. Un faltante de otro atributo no afecta esta decisión.

        Una categoría conocida en formato pero nunca observada en ese nodo devuelve
        las frecuencias del nodo alcanzado, en lugar de mezclar sus hijos. Si todos
        los atributos están ausentes, la mezcla recupera la distribución del nodo
        desde el cual se consulta, salvo redondeo. Esta operación no reentrena ni
        normaliza al final: predecir_probabilidades realiza esa normalización.
        """
        # Una hoja responde con las proporciones de clases que aprendió.
        if nodo.es_hoja:
            return [nodo.conteos.get(clase, 0.0) / nodo.peso_total for clase in self.classes_]
        valor = fila[nodo.atributo]

        # Un ausente combina todas las distribuciones, no solo las clases ganadoras.
        if es_faltante(valor):
            distribuciones_hijas = [
                (nodo.proporciones[rama], self._distribucion_fila(fila, hijo))
                for rama, hijo in nodo.hijos.items()]
            return [fsum(proporcion * distribucion[posicion]
                         for proporcion, distribucion in distribuciones_hijas)
                    for posicion in range(len(self.classes_))]

        # Una categoría nueva usa el resumen local; un conocido continúa por su rama.
        rama = ("<=" if valor <= nodo.umbral else ">") if nodo.tipo == "numerico" else valor
        if rama not in nodo.hijos:
            return [nodo.conteos.get(clase, 0.0) / nodo.peso_total for clase in self.classes_]
        return self._distribucion_fila(fila, nodo.hijos[rama])

    def _validar_prediccion(self, caracteristicas):
        """Comprueba que la consulta conserve las columnas y sus tipos aprendidos."""
        # Predecir requiere un árbol ya entrenado y la misma estructura de entrada.
        if self.raiz_ is None:
            raise RuntimeError("El árbol debe entrenarse antes de predecir.")
        self._validar_caracteristicas(caracteristicas)
        if not caracteristicas.columns.equals(self.columnas_):
            raise ValueError("La consulta debe conservar las columnas y el orden de entrenamiento.")
        for atributo, tipo in self.tipos_.items():
            if tipo == "numerico" and any(
                    not es_faltante(valor)
                    and (not isinstance(valor, Real) or isinstance(valor, bool))
                    for valor in caracteristicas[atributo]):
                raise ValueError(f"El atributo {atributo!r} debe conservar su tipo numérico.")

    def predecir_probabilidades(self, caracteristicas):
        """Devuelve un DataFrame de probabilidades para los casos consultados.

        Entrada: caracteristicas con las mismas columnas, orden y significado
        numérico del entrenamiento; admite valores ausentes. Requiere un árbol
        entrenado. Devuelve una fila por caso y una columna por clase en classes_,
        conservando exactamente índices, duplicados y orden, sin modificar la entrada.

        Combina las rutas necesarias y divide cada distribución por su suma para
        absorber el redondeo acumulado. Las filas suman uno y expresan frecuencias
        ponderadas, no probabilidades calibradas. Se leen tuplas para conservar el
        tipo individual de los valores. La validación puede lanzar RuntimeError
        si falta entrenamiento, TypeError por formato o ValueError por datos inválidos.
        """
        # Leer tuplas conserva el tipo de cada valor, incluidos enteros grandes.
        self._validar_prediccion(caracteristicas)
        distribuciones = [
            self._distribucion_fila(dict(zip(self.columnas_, valores)), self.raiz_)
            for valores in caracteristicas.itertuples(index=False, name=None)]

        # Corregir únicamente el redondeo acumulado de las sumas fraccionarias.
        distribuciones = [[probabilidad / fsum(fila) for probabilidad in fila]
                          for fila in distribuciones]
        return pd.DataFrame(distribuciones, columns=self.classes_.copy(),
                            index=caracteristicas.index.copy())

    def predecir(self, caracteristicas):
        """Devuelve un DataFrame con una única columna prediccion.

        Recibe la misma tabla y aplica las mismas validaciones que
        predecir_probabilidades. Elige la clase de mayor probabilidad por caso;
        un empate conserva la primera clase en classes_, es decir, la primera
        observada en el entrenamiento completo. Conserva índice y orden sin
        modificar la consulta ni reentrenar. Los nombres predict y predict_proba
        acceden a estas mismas operaciones.
        """
        # Elegir la clase de mayor probabilidad; el orden global resuelve empates.
        probabilidades = self.predecir_probabilidades(caracteristicas)
        clases = [self.classes_[max(range(len(fila)), key=fila.__getitem__)]
                  for fila in probabilidades.itertuples(index=False, name=None)]
        return pd.DataFrame({"prediccion": clases}, index=caracteristicas.index.copy())

    # Accesos compatibles al mismo código para entrenamiento e inferencia.
    fit = entrenar
    predict = predecir
    predict_proba = predecir_probabilidades
